### Get data from hoogspanningsnet.com

In [153]:

import geopandas as gpd
from shapely.geometry import Point
from collections import defaultdict
from itertools import product
from aocutils.special import UnionFind
from itertools import product
from copy import deepcopy
from functools import cache
import math


spanningscutoff = 100
use_tennet_stations = True
cutoff = 0.00001 # max distance between connections

# max_distance_line_station = 0.0005 # not too large since Borselle becomes ZKL
max_distance_line_station = 0.004 # not too large since Borselle becomes ZKL
max_distance = 0.006 # max distance between stations

accept_multiple_lines = False # for drawing
scale = 12 # the amount to scale with (higher --> more scaling down)
cluster_stations = True
spanning_kleuren = {
    380: 'red',
    150: 'blue',
    220: 'forestgreen',
    110: 'black',
    320: 'fuchsia',
    400: 'fuchsia'
}  

interconnectors = ["GNA-HGL380 W", "GNA-HGL380 Z", "DTC-NDR380 W", "DTC-NDR380 Z", "VYK-MBT380 W", "VYK-MBT380 Z", "RLL-ZVL380 G", "RLL-ZVL380 W", "MEE-DIL380 Z", "MEE-DIL380 W", "MBT-SDF380 Z", "MBT-OBZ380 W (SFK)", "EDC380-FDC300 Z"]

offshore = ["ZKL-BSA220 W", "ZKL-BSA220 O", "ZKL-BSB220 P", "HZL-HZB220 P", "HZL-HZA220 Z", "HZL-HZA220 W", "HNL-HNA220 Z", "HNL-HNA220 W", "HNL-HWA220 O", "HNL-HWA220 P", "WDC-EDR400 Nvt"]

In [154]:
def inspect_netschakel(netschaekl):
    for n in netschakels[netschaekl]:
        print(n, id2conn[n]['properties']['from_id'], id2conn[n]['properties']['to_id'])

In [155]:
# Laad de Nederland shapefile (EPSG:3035)
nl = gpd.read_file("netherlands_country_boundary/netherlands_Netherlands_Country_Boundary.shp")
country_shape = nl.unary_union  # merge all polygons once

def punt_in_nederland(lon, lat, boundary=country_shape):
    punt = Point(lon, lat)
    return boundary.contains(punt)

print(punt_in_nederland(4.895, 52.370))  # Amsterdam, True|
print(punt_in_nederland(6.14, 49.78))    # Buiten Nederland, False
print(punt_in_nederland(7.404059, 52.2884293))    # Buiten Nederland, False
print(punt_in_nederland(*[6.926667, 51.498889]))    # Buiten Nederland, False
print(punt_in_nederland(*[6.6308981, 51.0604542]))    # Buiten Nederland, False
print(punt_in_nederland(*[7.03359, 52.202448]))    # Buiten Nederland, False
print(punt_in_nederland(*[7.0325,52.2016]))    # Buiten Nederland, False

def distance_to_border(lon, lat, boundary=country_shape):
    punt = Point(lon, lat)
    if not boundary.contains(punt):
        return -punt.distance(boundary.boundary)
    else:
        return punt.distance(boundary.boundary)

def outside_border(lon, lat, boundary=country_shape, threshold=0.001, verbose=False):
    punt = Point(lon, lat)
    return not boundary.contains(punt)

# Compute centroids of Netherlands
x = float(nl.geometry.centroid.x.iloc[0])
y = float(nl.geometry.centroid.y.iloc[0])

True
False
False
False
False
False
False


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_17740\1331025071.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  country_shape = nl.unary_union  # merge all polygons once
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_17740\1331025071.py:29: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  x = float(nl.geometry.centroid.x.iloc[0])
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_17740\1331025071.py:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  y = float(nl.geometry.centroid.y.iloc[0])


In [156]:

import requests
import networkx as nx
import matplotlib.pyplot as plt
from math import sqrt
from urllib.parse import urlparse, parse_qs

In [157]:


url = 'https://webkaart.hoogspanningsnet.com/layerdata.php?type=trms&zoom=8&bbox=4.086914062500001%2C51.23784668914442%2C17.869262695312504%2C53.68044193408406'

# URL parsen
parsed = urlparse(url)
query = parse_qs(parsed.query)
bbox = "3.315,50.775,7.226,53.576"

# Stations ophalen (stic)
def get_url(t, zoom=14):
    url = f"https://webkaart.hoogspanningsnet.com/layerdata.php?type={t}&zoom={zoom}&bbox={bbox}"
    return requests.get(url).json()

def getconn(num):
    if use_tennet_stations:
        return next(f for f in verbindingen['features'] if f['properties']['ID'] == str(num))
    else:
        return next(f for f in verbindingen['features'] if f['properties']['ID'] == 'v'+str(num))

def findstation(naam):
    return next(s for s in hoogspanning_stations['features'] if s['properties']['Naam']==naam)
hoogspanning_stations = get_url('stic')
hoogspanning_verbindingen = get_url('trvb')
hoogspanning_polygons = get_url('sttr', zoom=14) # werkt alleen bij zoom = 14 of gedetailleerder
knooppunten = get_url('trkp', zoom=14) 
masten = get_url('trms', zoom=14) 

for var in [hoogspanning_stations, hoogspanning_verbindingen, hoogspanning_polygons, knooppunten, masten]:
    print(len(var['features']))


2033
7598
2991
193
30534


### Remove stations and lines that are not relevant

In [158]:


hoogspanning_stations['features'] = [f for f in hoogspanning_stations['features'] if punt_in_nederland(*f['geometry']['coordinates']) and f['properties']['Spanning']>= spanningscutoff]
hoogspanning_polygons['features'] = [f for f in hoogspanning_polygons['features'] if punt_in_nederland(*f['geometry']['coordinates'][0][0]) and f['properties']['Spanning']>= spanningscutoff]
hoogspanning_verbindingen['features'] = [f for f in hoogspanning_verbindingen['features'] if (punt_in_nederland(*f['geometry']['coordinates'][0]) or punt_in_nederland(*f['geometry']['coordinates'][-1])) and f['properties']['Spanning']>= spanningscutoff]
print(len(hoogspanning_stations['features']), len(hoogspanning_polygons['features']), len(hoogspanning_verbindingen['features']))

458 593 1446


### Alternatively load TenneT Geojsons

In [159]:
import json
from itertools import chain
with open("tennet/Hoogspanning_station.geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)
with open("tennet/Opstijgpunt.geojson", "r", encoding="utf-8") as f:
    geo2 = json.load(f)
  
tennet = {}
tennet['features'] = []
for feature in chain(geo['features']): #, geo2['features']):
    props = feature["properties"]
    if "SE_FLD55_SPANNINGSNIVEAU" in props:
        props["Spanning"] = props.pop("SE_FLD55_SPANNINGSNIVEAU")
    elif "SE_FLD18_SPANNINGSNIVEAU" in props:
        props["Spanning"] = props.pop("SE_FLD18_SPANNINGSNIVEAU")
        
    if "SE_FLD24_OBJECTOMSCHRIJVING" in props:
        props["Naam"] = props.pop("SE_FLD24_OBJECTOMSCHRIJVING")
        props["Osp"] = False
    elif 'SE_FLD13_OBJECTID' in props: # opstijgpunt
        props["Naam"] = props.pop("SE_FLD13_OBJECTID")
        props["Osp"] = True
        
        
    if feature["geometry"] is not None and props['Naam'] is not None:
        idx = props["Naam"].rfind(' ')
        # print(props["Naam"])
        if idx > 8 and props["Naam"].startswith('Station'): props["Naam"] = props["Naam"][8:idx]
        if props["Naam"][-1].isdigit():
            idx = props["Naam"].rfind(' ')
            props["Naam"] = props["Naam"][:idx]
        if props["Spanning"] > spanningscutoff or props['Naam'] == 'Oudehaske':
            tennet['features'].append(feature)
            if 'coordinates' not in feature['geometry']:
                print(feature)
with open("tennet/Hoogspanning_kabel_(ondergronds).geojson", "r", encoding="utf-8") as f:
    geo = json.load(f)
with open("tennet/Hoogspanning_leiding_(bovengronds).geojson", "r", encoding="utf-8") as f:
    geo2 = json.load(f)
tennet_verbindingen = {
    "type": "FeatureCollection",
    "features": geo["features"] + geo2["features"]
}
for feature in tennet_verbindingen['features']:
    feature['properties']['ID'] = feature['properties']['SE_FLD32_OBJECTID'] if 'SE_FLD32_OBJECTID' in feature['properties'] else feature['properties']['SE_FLD33_OBJECTID']
    feature['properties']['Spanning'] = feature['properties']['SE_FLD38_SPANNINGSNIVEAU'] if 'SE_FLD38_SPANNINGSNIVEAU' in feature['properties'] else feature['properties']['SE_FLD39_SPANNINGSNIVEAU']
    feature['properties']['Netschakel'] = feature['properties']['SE_FLD27_NETSCHAKELID'] if 'SE_FLD27_NETSCHAKELID' in feature['properties'] else feature['properties']['SE_FLD28_NETSCHAKELID']
    
tennet_verbindingen['features'] = [f for f in tennet_verbindingen['features'] if f['geometry']['type'] == 'LineString' and feature['properties']['Spanning'] > spanningscutoff]

In [160]:
for s in tennet['features']:
    if 'Oude' in s['properties']['Naam']:
        print(s['properties'])

{'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 1262304000000, 'ESRI_OID': 25, 'SE_FLD23_OBJECTID': 'EOS380', 'Shape__Area': 36061.02653503418, 'Shape__Length': 751.9155389363436, 'Spanning': 380, 'Naam': 'Eemshaven Oudeschip', 'Osp': False}
{'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 599616000000, 'ESRI_OID': 126, 'SE_FLD23_OBJECTID': 'ODL150', 'Shape__Area': 5023.072624206543, 'Shape__Length': 282.0907012690734, 'Spanning': 150, 'Naam': 'Oudeland', 'Osp': False}
{'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 852076800000, 'ESRI_OID': 127, 'SE_FLD23_OBJECTID': 'ODR150', 'Shape__Area': 6395.154697418213, 'Shape__Length': 326.62267415540947, 'Spanning': 150, 'Naam': 'Oudenrijn', 'Osp': False}
{'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf', 'SE_FLD4_BOUWJAAR': 189302400000, 'ESRI_OID': 324, 'SE_FLD23_OBJECTID': 'OHK220', 'Shape__Area': 28976.01528930664, 'Shape__Length': 807.4463518609955, 'Spanning': 220, 'Naam': 'Oudehaske', 'Osp': False}
{'SE_FLD1_

In [161]:
with open("tennet/Opstijgpunt.geojson", "r", encoding="utf-8") as f:
    geo2 = json.load(f)
osp = [o['geometry']['coordinates'][0][0] for o in geo2['features']]
len(osp)

161

### Determine stationsnamen of polygons

In [162]:
from math import sqrt, dist
from itertools import combinations
from copy import deepcopy

if use_tennet_stations:
    stations = deepcopy(tennet)
    verbindingen = deepcopy(tennet_verbindingen)
    verbindingen['features'] = [v for v in verbindingen['features'] if v['properties']['Spanning'] > spanningscutoff]
    station_polygons = deepcopy(tennet)
else:
    stations = deepcopy(hoogspanning_stations)
    verbindingen = deepcopy(hoogspanning_verbindingen)
    station_polygons = hoogspanning_polygons
    
id2conn = {f['properties']['ID']: f for f in verbindingen['features']}
not_found = {f['properties']['Naam'] for f in stations['features']}

if not use_tennet_stations:
    cnt = 1
    matched = 0
    for poly in station_polygons['features']:
        coords = poly['geometry']['coordinates'][0]
        avg_x = sum(p[0] for p in coords)/len(coords)
        avg_y = sum(p[1] for p in coords)/len(coords)
        poly_point = (avg_x, avg_y)
        
        # Vind het dichtstbijzijnde station
        min_dist = float('inf')
        closest_station_name = None
        for station in stations['features']:
            if use_tennet_stations:
                station_point = station['geometry']['coordinates'][0][0]
            else:
                station_point = station['geometry']['coordinates']
                
            d = dist(poly_point, station_point)
            if d < min_dist:
                min_dist = d
                closest_station = station
        
        if min_dist <= max_distance_line_station:
            poly['properties']['Naam'] = closest_station['properties']['Naam']
            not_found.discard(closest_station['properties']['Naam'])
            matched +=1 
        else:
            cnt +=1
    print(matched)
    print(len(not_found))
    not_found # Nuon Magnumcentrale is wel even interessant om in de gaten te houden



In [163]:
# 320 and 400

spanning = defaultdict(int)
for f in verbindingen['features']:
    spanning[f['properties']['Spanning']] += 1
spanning
    

defaultdict(int,
            {150: 42874, 110: 21805, 380: 21191, 220: 5939, 320: 33, 400: 7})

In [164]:
id2conn = {f['properties']['ID']: f for f in verbindingen['features']}
netschakels = defaultdict(list)
if use_tennet_stations:
    for v in verbindingen['features']:
        netschakels[v['properties']['Netschakel']].append(v['properties']['ID'])
        
coortonet = defaultdict(list)
for n in netschakels:
    for conn in netschakels[n]:
        c = id2conn[conn]['geometry']['coordinates'][0]
        coortonet[tuple(c)].append((n,conn))
        c = id2conn[conn]['geometry']['coordinates'][-1] # only the extremeties
        coortonet[tuple(c)].append((n,conn))
print(len(coortonet))
coortonet = {k:v for k,v in coortonet.items() if len(v)>1}
print(len(coortonet))
coortonet = {k:v for k,v in coortonet.items() if len(v)>2}
print(len(coortonet))

95238
88112
271


### Determine start and endlocation of connections

In [165]:


def point_in_polygon_strict(point, polygon):
    """
    Bepaal of een punt binnen een polygon ligt met ray-casting.
    point: (x, y)
    polygon: lijst van (x, y) tuples
    """
    x, y = point
    inside = False
    n = len(polygon)
    
    p1x, p1y = polygon[0]
    for i in range(n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y)*(p2x - p1x)/(p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

def point_line_distance(px, py, x1, y1, x2, y2):
    """Afstand van punt (px,py) tot lijnstuk (x1,y1)-(x2,y2)."""
    dx, dy = x2 - x1, y2 - y1
    if dx == dy == 0:
        return math.hypot(px - x1, py - y1)  # lijnstuk is een punt
    t = max(0, min(1, ((px - x1) * dx + (py - y1) * dy) / (dx*dx + dy*dy)))
    nx, ny = x1 + t*dx, y1 + t*dy
    return math.hypot(px - nx, py - ny)


def point_in_polygon_less_strict(point, polygon, epsilon=0.001):
    """
    Bepaal of een punt binnen een polygon ligt met ray-casting,
    of binnen 'epsilon' afstand van de rand.
    """
    x, y = point
    inside = False
    n = len(polygon)
    
    p1x, p1y = polygon[0]
    for i in range(n + 1):
        p2x, p2y = polygon[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        # check afstand tot rand
        if point_line_distance(x, y, p1x, p1y, p2x, p2y) <= epsilon:
            return True
        p1x, p1y = p2x, p2y

# Eerst bounding boxes van alle polygons berekenen
# Precompute bounding boxes and polygon lengths once
polygon_bboxes = []
for feature in station_polygons['features']:
    if 'Naam' in feature['properties']:
        coords = feature['geometry']['coordinates'][0]
        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
        polygon_bboxes.append({
            'ID': feature['properties']['Naam'],
            'min_x': min(xs),
            'max_x': max(xs),
            'min_y': min(ys),
            'max_y': max(ys),
            'polygon': coords,
            'spanning': feature['properties']['Spanning'],
            'n': len(coords)  # precompute once
        })

    
@cache
def find_polygon(x, y, epsilon=0.001):
    
    """Return the polygon ID containing the point, strict first, then epsilon margin"""
    # if use_tennet_stations: epsilon=0.000
    for bbox in polygon_bboxes:
        # Quick bounding box check
        if not (bbox['min_x'] <= x <= bbox['max_x'] and bbox['min_y'] <= y <= bbox['max_y']):
            continue  # skip polygons that can't contain the point
        # Strict point-in-polygon
        inside = False
        n = bbox['n']
        p1x, p1y = bbox['polygon'][0]
        for i in range(n + 1):
            p2x, p2y = bbox['polygon'][i % n]
            if y > min(p1y, p2y):
                if y <= max(p1y, p2y):
                    if x <= max(p1x, p2x):
                        if p1y != p2y:
                            xinters = (y - p1y)*(p2x - p1x)/(p2y - p1y) + p1x
                        if p1x == p2x or x <= xinters:
                            inside = not inside
            # less strict: check epsilon distance to edge
            dx, dy = p2x - p1x, p2y - p1y
            if dx == dy == 0:
                d = math.hypot(x - p1x, y - p1y)
            else:
                t = max(0, min(1, ((x - p1x) * dx + (y - p1y) * dy) / (dx*dx + dy*dy)))
                nx, ny = p1x + t*dx, p1y + t*dy
                d = math.hypot(x - nx, y - ny)
            if d <= epsilon:
                # print(d)
                return bbox['ID']
            p1x, p1y = p2x, p2y

        if inside:
            return bbox['ID']

    return False

connection_exclude = {'848719', '848727', '848731', '848734', '848725', '848717'} # Hengelo Marssteden has an overhead passing 380kV line

# Graaf opbouwen

# Controleer elk punt van elke verbinding
for feature in verbindingen['features']:
# for feature in [getconn(606969)]:

    conn_list = []
    # print(feature['properties']['ID'])
    verbinding_id = feature['properties']['ID']
    


    for point in feature['geometry']['coordinates']:
        if (res:=find_polygon(*point)):
            conn_list.append(res)
        
    if len(conn_list) > 1 and verbinding_id not in connection_exclude:
        color = spanning_kleuren.get(feature['properties']['Spanning'], 'grey')
    if verbinding_id not in connection_exclude:
        feature['properties']['from_id'] = find_polygon(*feature['geometry']['coordinates'][0])
        feature['properties']['to_id'] = find_polygon(*feature['geometry']['coordinates'][-1])
        feature['properties']['connections'] = set(conn_list)
    else:
        feature['properties']['from_id'] = False
        feature['properties']['to_id'] = False

    if len(set(conn_list)) > 2:
        pass
        # print(feature['properties']['ID'], feature['properties']['Spanning'],conn_list)

### at this point, not all lines and cables have a start and endstation, lets fix that

In [166]:


def parallellines(vid1, vid2):
    start1 = vid1['geometry']['coordinates'][0]
    end1 = vid1['geometry']['coordinates'][-1]
    start2 = vid2['geometry']['coordinates'][0]
    end2 = vid2['geometry']['coordinates'][-1]
    options = product([start1, end1], [start2, end2])
    return sum(dist(*o) < (cutoff * 1) for o in options) >= 2

In [167]:
def process_connections(results, netschakel):
    # if netschakel in interconnectors:
    #     print(netschakel, results)
    #     fds
    
    idcounter = 1
    found = []
    unique_connections = []
    for option in results:
        start = id2conn[option[0]]['properties']['from_id'] if id2conn[option[0]]['properties']['from_id'] else id2conn[option[0]]['properties']['to_id'] 
        end = id2conn[option[-1]]['properties']['from_id'] if id2conn[option[-1]]['properties']['from_id'] else id2conn[option[-1]]['properties']['to_id']
        if netschakel == 'HD-ZUV-LLS150 W':
            print('..', start, end)
        if start != end and sorted([start, end]) not in found:
            found.append(sorted([start, end]))
            unique_connections.append(deepcopy(id2conn[option[0]]))
            
            if not id2conn[option[0]]['properties']['from_id']: 
                unique_connections[-1]['geometry']['coordinates'] = unique_connections[-1]['geometry']['coordinates'][::-1]
            unique_connections[-1]['properties']['ID'] += '_' + str(idcounter)
            idcounter += 1
            unique_connections[-1]['properties']['from_id'] = start
            unique_connections[-1]['properties']['to_id'] = end
            for lineid in option[1:]:
                if id2conn[lineid]['geometry']['coordinates'][0] == unique_connections[-1]['geometry']['coordinates'][-1]:
                    unique_connections[-1]['geometry']['coordinates'] += id2conn[lineid]['geometry']['coordinates'][1:]
                else:
                    unique_connections[-1]['geometry']['coordinates'] += id2conn[lineid]['geometry']['coordinates'][::-1][1:]
    # print(len(results), len(unique_connections), found, [len(u['geometry']['coordinates']) for u in unique_connections] )
    if netschakel == 'HD-ZUV-LLS150 W':
            print('....', found)
            for x in unique_connections:
                print(x['properties'])
    return unique_connections
            

In [168]:
from operator import xor
from copy import deepcopy

def dfs(cur, path, goals, splits):
    results = []
    if id2conn[cur]['geometry']['coordinates'][0] in splits or id2conn[cur]['geometry']['coordinates'][-1] in splits:
        return [path]
    for neighbor in neigh[cur]:
        if neighbor not in path:
            if neighbor in goals:
                return [path + [neighbor]]
            else:
                path.append(neighbor)
                for r in dfs(neighbor, path, goals, splits):
                    results.append(r)
                assert path.pop() == neighbor
    # if '714529' in path: print(results, path)
    return results

            
forbidden_split = [[5.469404028332668,52.33157631469825],
                   [4.67473246782226, 52.4597878584342]
                 ]
forbidden_split = osp
def too_close_to_forbidden_splits(c):
    # return False
    # print(c)
    return min([dist(f, c) for f in forbidden_split]) < max_distance
# too_close_to_forbidden_splits([4.67473246782226, 52.4597878584342])
    
# make a dict
def make_coor2id(netschakel):
    coor2id = defaultdict(list)
    for lineid in netschakels[netschakel]:
        if id2conn[lineid]['properties']['from_id'] and id2conn[lineid]['properties']['to_id']:
            # print('line ignored since within a station', id2conn[lineid]['properties']['from_id'], id2conn[lineid]['properties']['to_id'])
            continue
        
        
        coor2id[tuple(id2conn[lineid]['geometry']['coordinates'][0])].append(lineid)
        coor2id[tuple(id2conn[lineid]['geometry']['coordinates'][-1])].append(lineid)
    return coor2id

def make_neighbors(coor2id):
    conn = defaultdict(set)
    for connected in coor2id.values():
        for c1 in connected:
            for c2 in connected:
            
                if c1 != c2:
                    conn[c1].add(c2)
    return conn

In [169]:

presentstations = {s['properties']['Naam'] for s in stations['features']}
seen = {}
splitcounter = 0
interconnectcounter = 9999
finalset = {}
finalset['features'] = []
for netschakel in netschakels:
    splitcounter += 1
    results = []
    coor2id = make_coor2id(netschakel) # per coordinate al the lines
    # for k,v in coor2id.items():
        # if len(v) > 2:
        #     print(netschakel, k, len(v), id2conn[v[0]]['properties']['from_id'], id2conn[v[0]]['properties']['to_id'])
    neigh = make_neighbors(coor2id) # per line all it's neighbors (on both sides)
    splits = [k for k,v in coor2id.items() if len(v) > 2 and not too_close_to_forbidden_splits(k)]
    
    if netschakel in (interconnectors + offshore) and str(interconnectcounter) not in presentstations:
        print('interconnect', netschakel)
        
        distances = []
        for line in netschakels[netschakel]:
            coor = id2conn[line]['geometry']['coordinates'][0]
            distances.append((distance_to_border(*coor), coor, line, 'from_id'))
            coor = id2conn[line]['geometry']['coordinates'][-1]
            distances.append((distance_to_border(*coor), coor, line, 'to_id'))
        distances.sort()
        furthest_dist, furthest_coor, furthest_line, fromto = distances[0]
        
        presentstations.add(str(interconnectcounter))        
        stations['features'].append(deepcopy(stations['features'][0]))
        stations['features'][-1]['geometry']['coordinates'] = [[furthest_coor]]
        stations['features'][-1]['properties']['Naam'] = str(interconnectcounter)
        stations['features'][-1]['properties']['Spanning'] = 380
        
        id2conn[furthest_line]['properties'][fromto] = str(interconnectcounter)
        
        print(netschakel, furthest_coor, furthest_line, fromto, furthest_dist)
        interconnectcounter -= 1
        
        
        
        
        
        
    # get starting lines
    for lineid in netschakels[netschakel]:
        line = id2conn[lineid]
        start = line['properties']['from_id']
        end = line['properties']['to_id']  
        # determine if line are starting points within a station (not fully within a station)  
        if line['geometry']['coordinates'][0] in splits:
            line['properties']['from_id'] = str(splitcounter)
        if line['geometry']['coordinates'][-1] in splits:
            line['properties']['to_id'] = str(splitcounter)
        if splits and str(splitcounter) not in presentstations:   
            presentstations.add(str(splitcounter))        
            stations['features'].append(deepcopy(stations['features'][0]))
            stations['features'][-1]['geometry']['coordinates'] = [[splits[0]]]
            stations['features'][-1]['properties']['Naam'] = str(splitcounter)
            stations['features'][-1]['properties']['Spanning'] = 380
        

    # need to do this after above
    starting = []
    for lineid in netschakels[netschakel]:
        line = id2conn[lineid]
        start = line['properties']['from_id']
        end = line['properties']['to_id']  
        if xor(start is False, end is False): 
            starting.append(lineid)


    for startinglineid in starting:
        results += dfs(startinglineid, [startinglineid], starting, splits)
        
    if len(results) == 0 and len(coor2id) > 50 and len(starting) >= 6:
        pass

    finalset['features'] += process_connections(results, netschakel)
         
print(len(finalset['features']), len(stations['features']))

interconnect HZL-HZA220 Z
HZL-HZA220 Z [4.04315694839766, 52.319317806873] 100020795 to_id -0.2890131762683734
interconnect HZL-HZA220 W
HZL-HZA220 W [4.04304868636369, 52.3193430914649] 100020958 to_id -0.2891054372306967
interconnect HZL-HZB220 P
HZL-HZB220 P [4.08484229357901, 52.2578760247719] 100023013 to_id -0.21585323955375793
interconnect WDC-EDR400 Nvt
WDC-EDR400 Nvt [8.40412450027104, 55.3852913421766] 100014387 from_id -2.3864586710284703
interconnect ZKL-BSB220 P
ZKL-BSB220 P [2.96578478931527, 51.7266411687083] 100016476 to_id -0.4370114849181477
interconnect ZKL-BSA220 W
ZKL-BSA220 W [3.05691586156931, 51.7000063022669] 100016763 to_id -0.3435812032871073
interconnect EDC380-FDC300 Z
EDC380-FDC300 Z [6.39143511838079, 53.669567079718] 100022290 to_id -0.10998834960387086
interconnect HNL-HNA220 Z
HNL-HNA220 Z [4.29373247156772, 52.6978671560763] 100027688 to_id -0.3186011489916803
interconnect HNL-HNA220 W
HNL-HNA220 W [4.29368250842537, 52.6978860850515] 100027633 from_i

### Now we know start and end locations and need to merge connecting connections with DFS (eg a line and a cable)

### Count the stations how many connections they have

In [170]:
verbindingen = finalset

In [171]:
if cluster_stations:
    from collections import defaultdict
    conn = defaultdict(int)
    for v in verbindingen['features']:
        conn[v['properties']['from_id']] += 1
        conn[v['properties']['to_id']] += 1
    conn

In [172]:

priority = ['Eemshaven', 'Borssele', 'Diemen', 'Geervliet']
station2spanning = {s['properties']['Naam']: s['properties']['Spanning'] for s in stations['features']}
if cluster_stations:
    from collections import defaultdict
    conn = defaultdict(int)
    for v in verbindingen['features']:
        conn[v['properties']['from_id']] += 1
        conn[v['properties']['to_id']] += 1
    conn


    ufstation = UnionFind([])
    for idx, s1 in enumerate(stations['features']):
        for idx2, s2 in enumerate(stations['features'][idx+1:], start=idx+1):
            if use_tennet_stations:
                if dist(s1['geometry']['coordinates'][0][0], s2['geometry']['coordinates'][0][0]) < max_distance:
                    ufstation.union(s1['properties']['Naam'],s2['properties']['Naam'])
            else:
                if dist(s1['geometry']['coordinates'], s2['geometry']['coordinates']) < max_distance:
                    ufstation.union(s1['properties']['Naam'],s2['properties']['Naam'])
    station2mainstation = {}
    removed = set()

    for g in ufstation.groups():
        sortedgroup = sorted(g, key=lambda x: (x not in priority, x[0].isnumeric(), -station2spanning[x], -conn[x]))
        print(sortedgroup)
        for s in sortedgroup[1:]:
            station2mainstation[s] = sortedgroup[0]
            removed.add(s)
        
else:
    removed = set()
    station2mainstation = {}

['Borssele', 'Zeeuwse Kust Landstation']
['Hengelo Weideweg']
['Vierverlaten']
['Lelystad']
['Diemen', 'Diemer Vijfhoek']
['Ens']
['Krimpen a/d IJssel']
['Eemshaven', 'Eemshaven Oudeschip', 'Eemshaven Converterstation 380 kV', 'Eemshaven Synergieweg 380kV', 'Waddenweg Converterstation 380 kV', 'Eemshaven Comp. en Filteren', 'Robbenplaat', 'Oostpolder', 'Eemshaven Oost']
['Groningen Hunze', 'Groningen Bornholmstraat']
['Bergum']
['Louwsmeer']
['Doetinchem', 'Langerak']
['Wijk aan Zee', 'Hollandse Kust Noord Landstation']
['Rilland', '157']
['Zeyerveen']
['Meeden']
['Westerlee', 'De Lier']
['Zwolle', 'Hessenweg', 'Zwolle Hessenweg']
['Borculo', 'Borculo Berkel']
['Beersdal', 'Huskensweg']
['Boekend', '825']
['Geervliet', 'Geervliet Noorddijk']
['Boxmeer', '871']
['Breukelen Kortrijk']
['Hengelo', 'Hengelo Oele']
['Amsterdam Noord Klaprozenweg', 'Amsterdam Noord Papaverweg']
['Wateringen']
['Anna Paulowna', 'BBL Gasunie']
['Vijfhuizen']
['Maasvlakte']
['Eindhoven', 'Eindhoven Oost']
['Bij

### Make data for loom

In [173]:
f = 'Geertruidenberg'
for edge in verbindingen['features']:
    if edge['properties']['from_id'] == f or edge['properties']['to_id']  == f:
        print(edge['properties']['Spanning'], edge['properties']['from_id'], edge['properties']['to_id'], edge['properties']['ID'])

150 Geertruidenberg Waalwijk 1075271_1
150 Geertruidenberg Biesbosch 1023944_1
150 Biesbosch Geertruidenberg 1144922_1
380 Rilland Geertruidenberg 2227947_1
380 Rilland Geertruidenberg 2227914_1
150 Geertruidenberg Oosteind 738863_1
150 Oosteind Geertruidenberg 738044_1
150 Geertruidenberg Breda 982184_1
150 Moerdijk Geertruidenberg 797891_1
150 Moerdijk Geertruidenberg 797750_1
150 Geertruidenberg Breda 982206_1
380 Geertruidenberg Eindhoven 516254_1
380 Eindhoven Geertruidenberg 555021_1
380 Eindhoven Geertruidenberg 555406_1
380 Krimpen a/d IJssel Geertruidenberg 513690_1
380 Krimpen a/d IJssel Geertruidenberg 513704_1


In [174]:
forbiddenlines = "2387372_1 2387343_2 548828_1 495468_1 495462_3 263392_2 1706983_1 1940482_3 1706977_2 742566_1 1709656_1 1744529_1 1621460_3 1621455_3".split()

In [175]:
scale = 11
verbindingen['features'] = sorted(verbindingen['features'], key = lambda x: -x['properties']['Spanning'])
forbidden = ['Enecogen']
forbidden = []
def normalize(coor, scale = scale):
    dx = coor[0] - x
    dy = coor[1] - y
    
    return [x + dx/scale, y + dy/scale]
   
from pyproj import Transformer

# Example: WGS84 (lon/lat) → UTM zone 31N (meters)
proj_to_meters = Transformer.from_crs("EPSG:4326", "EPSG:32631", always_xy=True)
proj_to_lonlat = Transformer.from_crs("EPSG:32631", "EPSG:4326", always_xy=True)

# Original normalize function in lon/lat
def normalize(coor, scale=scale):
    # 1. Convert lon/lat to meters
    mx, my = proj_to_meters.transform(coor[0], coor[1])
    mx0, my0 = proj_to_meters.transform(x, y)
    
    # 2. Do your scaling in meters
    dx = mx - mx0
    dy = my - my0
    mx_new = mx0 + dx / scale
    my_new = my0 + dy / scale
    
    # 3. Convert back to lon/lat
    lon_new, lat_new = proj_to_lonlat.transform(mx_new, my_new)
    return [lon_new, lat_new]
 
    
accepted = {f['properties']['Naam'] for f in stations['features'] if f['properties']['Naam'] not in forbidden} 

loom_nodes = {
    "type": "FeatureCollection",
    "features": []
}

for node in stations['features']:
    if node['properties']['Naam'] in accepted:
        loom_node = {
            "type": "Feature",
            "geometry": node['geometry'].copy(),
            "properties": {
                "id": node['properties']['Naam'],
                "station_label": ' ' + node['properties']['Naam'] + '  '
            }
        }
        # for d in ['deg', 'deg_in', 'deg_out']:
        #     loom_node['properties'][d] =G.degree[node['properties']['Naam']]
        if use_tennet_stations:
            loom_node["geometry"]["coordinates"] = normalize(loom_node["geometry"]["coordinates"][0][0])
            loom_node['geometry']['type'] = 'Point'
        else:
            loom_node["geometry"]["coordinates"] = normalize(loom_node["geometry"]["coordinates"])
            
        loom_nodes['features'].append(loom_node)

# Save to JSON
with open("loom_nodes.json", "w") as f:
    json.dump(loom_nodes, f, indent=2)

print("Conversion done! Saved as loom_nodes.json")

# Convert edges to Loom format
loom_edges = {
    "type": "FeatureCollection",
    "features": []
}

def sample(lst, size=10):
    newlist = lst[::size]
    if newlist[-1] != lst[-1]:
        newlist.append(lst[-1])
    return newlist

seen = set()
for edge in verbindingen['features']:
    if edge['properties']['ID'] == '2387372':
        print(edge['properties'])
    if 'from_id' in edge['properties'] and edge['properties']['to_id'] and edge['properties']['Spanning'] > spanningscutoff and edge['properties']['ID'] not in forbiddenlines:
        
        spanning = str(round(edge['properties']['Spanning']))
        loom_edge = {
            "type": "Feature",
            "geometry": edge['geometry'].copy(),
            "properties": {
                "identifier": edge['properties']['ID'],
                "from": station2mainstation.get(edge['properties']['from_id'], edge['properties']['from_id']),
                "to": station2mainstation.get(edge['properties']['to_id'], edge['properties']['to_id']),
                "dbg_lines": str(round(edge['properties']['Spanning'])),
                "spanning": spanning,
                "lines": [{
                "color": spanning_kleuren.get(int(spanning), 'grey'),
                "id": spanning,
                # "id": edge['properties']['ID'],
                # "label": spanning
                }]
                
                # optional: add a label, color, etc.
            }
        }
        
        loom_edge["geometry"]["coordinates"] = [normalize(c) for c in loom_edge["geometry"]["coordinates"]]
        # loom_edge["geometry"]["coordinates"] = loom_edge["geometry"]["coordinates"][:25] + loom_edge["geometry"]["coordinates"][-25:]
        loom_edge["geometry"]["coordinates"] = [loom_edge["geometry"]["coordinates"][0], loom_edge["geometry"]["coordinates"][-1]]
        
        # loom_edge["geometry"]["coordinates"] = sample(loom_edge["geometry"]["coordinates"], 10)

            
        # if loom_edge['properties']['from'] in accepted and loom_edge['properties']['to'] in accepted: 
        if True:
            # print(loom_edge['properties'])
            if loom_edge['properties']['from'] != loom_edge['properties']['to']: # no self loops
                loom_edge["properties"]["fromto"] = tuple(sorted((loom_edge['properties']['from'], loom_edge['properties']['to'], loom_edge['properties']['spanning'])))
                if loom_edge["properties"]["fromto"] not in seen:
                    seen.add(loom_edge["properties"]["fromto"])
                    loom_edges['features'].append(loom_edge)
                else:
                    if accept_multiple_lines:
                        for e in loom_edges['features']:
                            if e['properties']['fromto'] == loom_edge['properties']['fromto']:
                                e['properties']['lines'].append(loom_edge['properties']['lines'][0])
                                break
        else:
            pass
        
    else:
        pass
        # print('not found')
# Save to JSON
with open("loom_edges.json", "w") as f:
    json.dump(loom_edges, f, indent=2)

print("Edges converted to Loom format!")

loom_all = {
    "type": "FeatureCollection",
    "features": loom_nodes['features'] + loom_edges['features']
}

# Optionally save to a file
with open("loom_combined.json", "w") as f:
    json.dump(loom_all, f, indent=2)
from pathlib import Path

wsl_path = Path(r"\\wsl.localhost\Ubuntu-24.04\home\jesse\loom\examples\netkaart3.json")

# Write JSON with LF line endings
with wsl_path.open("w", encoding="utf-8", newline="\n") as f:
    json.dump(loom_all, f, indent=2, ensure_ascii=False)
    f.write("\n")  # make sure file ends with a newline
print("Nodes and edges combined into one Loom JSON!")

# cat examples/netkaart3.json | docker run -i loom loom | docker run -i loom octi | docker run -i loom transitmap -l > netkaart-octilinear.svg

Conversion done! Saved as loom_nodes.json
Edges converted to Loom format!
Nodes and edges combined into one Loom JSON!


### Deploy

In [176]:
fds

NameError: name 'fds' is not defined

In [ ]:
# Optimal settings for big map

# import geopandas as gpd
# from shapely.geometry import Point
# import json
# from collections import defaultdict
# spanningscutoff = 100
# use_tennet_stations = True
# cutoff = 0.00001 # max distance between connections
# # max_distance_line_station = 0.0005 # not too large since Borselle becomes ZKL
# max_distance_line_station = 0.004 # not too large since Borselle becomes ZKL
# max_distance = 0.006 # max distance between stations

# accept_multiple_lines = False # for drawing
# scale = 11 # the amount to scale with (higher --> more scaling down)
# cluster_stations = True
# spanning_kleuren = {
#     380: 'red',
#     150: 'blue',
#     220: 'forestgreen',
#     110: 'black'
# }  

# cat examples/netkaart3.json | ./build/topo | ./build/loom | ./build/octi -g 400 --geo-pen 0.1 --nd-move-pen 0.5 | ./build/transitmap -l --line-width 50 --station-label-textsize 300 > 0finalgeopen01scale11b.svg

In [ ]:
spanningscutoff = 100
use_tennet_stations = True
cutoff = 0.00001 # max distance between connections
# max_distance_line_station = 0.0005 # not too large since Borselle becomes ZKL
max_distance_line_station = 0.004 # not too large since Borselle becomes ZKL
max_distance = 0.01 # max distance between stations

accept_multiple_lines = False # for drawing
scale = 10 # the amount to scale with (higher --> more scaling down)
cluster_stations = True
spanning_kleuren = {
    380: 'red',
    150: 'blue',
    220: 'forestgreen',
    110: 'black'
}  

cat examples/netkaart3.json | ./build/topo | ./build/loom | ./build/octi -g 400 --geo-pen 0.5--nd-move-pen 1 | ./build/transitmap -l --line-width 50 --station-label-textsize 300 > goallnewlogic.svg

SyntaxError: invalid syntax (2090385435.py, line 18)

In [ ]:
# for the big map
# cat examples/netkaart3.json | ./build/topo | ./build/loom | ./build/octi -g 400 --geo-pen 0--nd-move-pen 0 | ./build/transitmap -l --line-width 50 --station-label-textsize 300 > afinalfornowscale10.svg

In [ ]:

docker run -it \
  -v "$PWD":/workspace \
  -v /mnt/c/Users/Gebruiker/AppData/Local/Microsoft/Windows/Fonts:/usr/share/fonts/truetype/windows:ro \
  -w /workspace \
  loom-dev bash


apt update
apt install -y fontconfig

mkdir -p build
cd build
cmake ..
make -j"$(nproc)"
cd ..

SyntaxError: invalid syntax (4011303740.py, line 1)

In [ ]:
# after updating source
cd build && make -j$(nproc) transitmap && cd .. && cat examples/netkaart3.json | ./build/loom | ./build/octi -g 400 --nd-move-pen 0.1 | ./build/transitmap -l --line-width 50 --station-label-textsize 300 > go2newlogic.svg

In [ ]:
# outside container
cat examples/netkaart3.json | docker run -i loom topo | docker run -i loom loom | docker run -i loom octi -g 400 --nd-move-pen 1 | docker run -i loom transitmap -l --line-width 50 --station-label-textsize 150  > netkaart-octilinear400-17pen.svg

# inside container
cat examples/netkaart3.json | ./build/loom | ./build/octi -g 400 --nd-move-pen 1 | ./build/transitmap -l --line-width 30 --station-label-textsize 150 > netkaart-svg.svg

KeyError: 'Simonshaven'

### Debug

hi9
hi9
hi9


In [ ]:
next(s for s in stations['features'] if s['properties']['Naam']=='587')

{'type': 'Feature',
 'id': 3,
 'geometry': {'type': 'Polygon',
  'coordinates': [[[4.67473246782226, 52.4597878584342]]]},
 'properties': {'SE_FLD1_BEDRIJFSSTATUS': 'In bedrijf',
  'SE_FLD4_BOUWJAAR': 94694400000,
  'ESRI_OID': 3,
  'SE_FLD23_OBJECTID': 'NIWG150',
  'Shape__Area': 8239.912483215332,
  'Shape__Length': 372.6012611624378,
  'Spanning': 380,
  'Naam': '587',
  'Osp': False}}

In [ ]:
std::string color = c.geoms[i].from.line->color();
if (color == "FF0000") {       // red
    strokeWidth *= 3;
} else if (color == "228B22") { // green
    strokeWidth *= 2.25;
} else if (color == "0000FF") { // blue
    strokeWidth *= 1.5;
}





      std::stringstream styleOutlineCropped;
      styleOutlineCropped << "fill:none;stroke:#000000";

      styleOutlineCropped << ";stroke-linecap:butt;stroke-width:"
                          << (_cfg->lineWidth + _cfg->outlineWidth) *
                                 _cfg->outputResolution;
      Params paramsOutlineCropped;
      paramsOutlineCropped["style"] = styleOutlineCropped.str();
      paramsOutlineCropped["class"] += " inner-geom-outline";
      paramsOutlineCropped["class"] +=
          " " + getLineClass(c.geoms[i].from.line->id());

std::stringstream styleStr;
styleStr << "fill:none;stroke:#" << color;
styleStr << ";stroke-linecap:round;stroke-opacity:1;stroke-width:" 
         << strokeWidth;
         

In [ ]:
forbidden = {
 'AKU1 (Emmtec)',
 'AL Stoom',
 'AMS 13/14',
 'AMS98',
 'APN',
 'Aardgasbuffer Zuidwending',
 'Agriport-A7',
 'Air Liquide',
 'Aldel (Heveskes)',
 'Amer 6',
 'Amer 7',
 'Amer 9',
 'Ampyr',
 'Attero',
 'Attero Moerdijk',
 'Bergumcentrale',
 'Borssele 30',
 'Borssele Scaldia',
 'Byron Jackson Flowserve',
 'Centrale Moerdijk',
 'Claus A',
 'Claus C',
 'Claus C4',
 'DSM-1 Swentibold',
 'DSM-2 Kerensheide',
 'DSM-3 Neerbeek',
 'DSM-4 Oude Postbaan',
 'Delesto DES-I',
 'Delesto DES-II',
 'Delfzijl Farmsum (Golden Raand)',
 'Dintelhaven Betuweroute',
 'EC20',
 'EC3',
 'EC4',
 'EC5',
 'EC6',
 'EC7',
 'ECL Luttelgeest',
 'ECW Wieringermeer',
 'EH10',
 'EH20',
 'EH30',
 'ELSTA GTG-301',
 'EPNL Rijnmond 1',
 'EPNL Rijnmond II',
 'EPZ EB',
 'ESSO Botlek',
 'EdgeConnex',
 'Eemshaven COBRA',
 'Eemshaven Midden',
 'Eemshaven NorNed',
 'Eemshaven filter',
 'Eindhovencentrale',
 'Eneco Lage Weide 6',
 'Eneco MK12',
 'Enexis Dongecentraleweg',
 'GD Groen',
 'Gasunie Bacton-Balgzand',
 'Gasunie CS Scheemda',
 'Gemini',
 'Google Nimble',
 'Graafstroom Betuweroute',
 'Grijpskerk UGS',
 'HKN Landstation',
 'HKZ-Landstation',
 'HVDC BritNed',
 'Hemweg HW9',
 'Hengelo Salinco',
 'Hydro Agri (Schakelaars)',
 'Hydro Agri Sluiskil',
 'Klantstation Allnex',
 'MPP3',
 'MVL Enecogen',
 'MVL Onyx',
 'Meeden DRT',
 'Moerdijk CCGT',
 'Màximacentrale',
 'NAM Delfzijl Schaapbulten',
 'NAM Menterwolde-Spitsbergen',
 'NAM Scheemda De Eeker',
 'NAM Scheemderzwaag',
 'NAM Schoonebeek',
 'NAM Slochteren Kooipolder',
 'NLR',
 'NOP-Agrowind',
 'NUON Velsen24',
 'NUON Velsen25 PER',
 'NXP Philips',
 'Nobian',
 'NoordZeeWind/OWEZ',
 'Norg UGS',
 'NorthC Blackbox',
 'Nuon Magnum',
 'Oostpolder (Saturn)',
 'PerGen',
 'RWE-EC30',
 'RWE-EC31',
 'Reeweg Havenspoorlijn',
 'RoCa Centrale',
 'Shell Moerdijk',
 'Shell SHH1',
 'Shell shunt',
 'Slochteren Dellerweerden',
 'Sloe (Vlissingen Oost)',
 'Sloe I',
 'Sloe II',
 'Stikstoffabriek Zuidbroek',
 'Swentibold',
 'TAQA (Boekelermeer Zuid)',
 'TATA HSV23',
 'TATA HSV26',
 'TATA HSV28',
 'TATA HVS20',
 'Twence AVI',
 'Twence biomassa',
 'Urenco',
 'Vattenfall Centrale Almere',
 'Vattenfall IJmond 01',
 'Vogelweg HV',
 'WKC Helmond 1/2',
 'WP-Zuidwester',
 'Windpark Blauw',
 'Windpark Bouwdokken',
 'Windpark Friesland',
 'Windpark Krammer',
 'Windpark N33',
 'Windpark Zuidlob',
 'Zeeland Refinery',
 'Zoetermeer-HSL',
 'Zonnepark HVC',
 'Zonnepark Midden Groningen',
 'Zonnepark Stadskanaal'}

In [ ]:
// Copyright 2016, University of Freiburg,
// Chair of Algorithms and Data Structures.
// Authors: Patrick Brosi <brosi@informatik.uni-freiburg.de>

#include <stdint.h>

#include <fstream>
#include <ostream>

#include "shared/linegraph/Line.h"
#include "shared/rendergraph/RenderGraph.h"
#include "transitmap/config/TransitMapConfig.h"
#include "transitmap/label/Labeller.h"
#include "transitmap/output/SvgRenderer.h"
#include "util/String.h"
#include "util/geo/PolyLine.h"
#include "util/log/Log.h"
using shared::linegraph::Line;
using shared::linegraph::LineNode;
using shared::rendergraph::InnerGeom;
using shared::rendergraph::RenderGraph;
using transitmapper::label::Labeller;
using transitmapper::output::InnerClique;
using transitmapper::output::SvgRenderer;
using util::geo::DPoint;
using util::geo::DPolygon;
using util::geo::LinePoint;
using util::geo::LinePointCmp;
using util::geo::Polygon;
using util::geo::PolyLine;

// _____________________________________________________________________________
SvgRenderer::SvgRenderer(std::ostream* o, const config::Config* cfg)
    : _o(o), _w(o, true), _cfg(cfg) {}

// _____________________________________________________________________________
void SvgRenderer::print(const RenderGraph& outG) {
  std::map<std::string, std::string> params;
  RenderParams rparams;

  auto box = outG.getBBox();

  box = util::geo::pad(
      box, outG.getMaxLineNum() * (_cfg->lineWidth + _cfg->lineSpacing));

  Labeller labeller(_cfg);
  if (_cfg->renderLabels) {
    LOGTO(DEBUG, std::cerr) << "Rendering labels...";
    labeller.label(outG, _cfg->dontLabelDeg2);
    box = util::geo::extendBox(labeller.getBBox(), box);
  }

  double p = _cfg->outputPadding;

  box = util::geo::pad(box, p);

  if (!_cfg->worldFilePath.empty()) {
    std::ofstream file;
    file.open(_cfg->worldFilePath);
    if (file) {
      file << 1 / _cfg->outputResolution << std::endl
           << 0 << std::endl
           << 0 << std::endl
           << -1 / _cfg->outputResolution << std::endl
           << std::fixed << box.getLowerLeft().getX() << std::endl
           << box.getUpperRight().getY() << std::endl;
      file.close();
    }
  }

  rparams.xOff = box.getLowerLeft().getX();
  rparams.yOff = box.getLowerLeft().getY();

  rparams.width = box.getUpperRight().getX() - rparams.xOff;
  rparams.height = box.getUpperRight().getY() - rparams.yOff;

  rparams.width *= _cfg->outputResolution;
  rparams.height *= _cfg->outputResolution;

  auto latLngLL = util::geo::webMercToLatLng<double>(box.getLowerLeft().getX(),
                                                     box.getLowerLeft().getY());
  auto latLngUR = util::geo::webMercToLatLng<double>(
      box.getUpperRight().getX(), box.getUpperRight().getY());

  params["latlng-box"] = std::to_string(latLngLL.getX()) + "," +
                         std::to_string(latLngLL.getY()) + "," +
                         std::to_string(latLngUR.getX()) + "," +
                         std::to_string(latLngUR.getY());

  params["width"] = std::to_string(rparams.width);
  params["height"] = std::to_string(rparams.height);
  params["viewBox"] = "0 0 " + std::to_string(rparams.width) + " " +
                      std::to_string(rparams.height);
  params["xmlns"] = "http://www.w3.org/2000/svg";
  params["xmlns:xlink"] = "http://www.w3.org/1999/xlink";

  *_o << "<?xml version=\"1.0\" encoding=\"UTF-8\"?>\n";
  *_o << "<!DOCTYPE svg PUBLIC \"-//W3C//DTD SVG 1.1//EN\" "
         "\"http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd\">";

  LOGTO(DEBUG, std::cerr) << "Rendering edges...";
  if (_cfg->renderEdges) {
    outputEdges(outG, rparams);
  }
  _w.openTag("svg", params);

  _w.openTag("defs");

  LOGTO(DEBUG, std::cerr) << "Rendering markers...";
  for (auto const& m : _markers) {
    params.clear();
    params["id"] = m.name;
    params["orient"] = "auto";
    params["markerWidth"] = "20";
    params["markerHeight"] = "4";
    params["refY"] = "0.5";
    params["refX"] = "0";

    _w.openTag("marker", params);

    params.clear();
    params["d"] = m.path;
    params["fill"] = m.color;
    ;

    _w.openTag("path", params);

    _w.closeTag();
    _w.closeTag();
  }

  _w.closeTag();

  LOGTO(DEBUG, std::cerr) << "Rendering nodes...";
  for (auto n : outG.getNds()) {
    if (_cfg->renderNodeConnections) {
      renderNodeConnections(outG, n, rparams);
    }
  }

  LOGTO(DEBUG, std::cerr) << "Writing edges...";
  renderDelegates(outG, rparams);

  LOGTO(DEBUG, std::cerr) << "Writing nodes...";
  outputNodes(outG, rparams);
  if (_cfg->renderNodeFronts) {
    renderNodeFronts(outG, rparams);
  }

  LOGTO(DEBUG, std::cerr) << "Writing labels...";
  if (_cfg->renderLabels) {
    renderLineLabels(labeller, rparams);
    renderStationLabels(labeller, rparams);
  }

  _w.closeTags();
}

// _____________________________________________________________________________
// # this function is changed
void SvgRenderer::outputNodes(const RenderGraph& outG,
                              const RenderParams& rparams) {
    _w.openTag("g");
    for (auto n : outG.getNds()) {
        std::map<std::string, std::string> params;

        if (_cfg->renderStations && n->pl().stops().size() > 0 &&
            n->pl().fronts().size() > 0) {
            params["stroke"] = "black";
            params["stroke-width"] =
                util::toString((_cfg->lineWidth / 2) * 0.6 * _cfg->outputResolution); //changed from 2
            params["fill"] = "white";

            for (const auto& geom : outG.getStopGeoms(n, _cfg->tightStations, 32)) {
                // create a copy to scale
                Polygon<double> scaledGeom = geom;

                // compute center manually
                DPoint center(0, 0);
                auto& pts = geom.getOuter();
                for (const auto& p : pts) {
                    center.setX(center.getX() + p.getX());
                    center.setY(center.getY() + p.getY());
                }
                center.setX(center.getX() / pts.size());
                center.setY(center.getY() / pts.size());

                // scale each vertex 4× relative to the center
                for (auto& p : scaledGeom.getOuter()) {
                    double dx = p.getX() - center.getX();
                    double dy = p.getY() - center.getY();
                    p.setX(center.getX() + dx * 1); // changed from 2
                    p.setY(center.getY() + dy * 1); // changed from 2
                }
                // const std::string& stationId = n->id;
                // const std::string& stationId = n->pl().name;
                // if (std::isdigit(n->getDeg()[0])) {
                  // continue;  // skip this geometry
                // }
                // # added
                const auto& station = n->pl().stops().front();  // Station object
                const std::string& stationName = station.name;      // string
                if (stationName.size() >= 1 && std::isdigit(stationName[2])) {
                    std::cerr << "Station ID: " << stationName << std::endl;
                    continue;  // skip this geometry
                }
                // end added
                printPolygon(scaledGeom, params, rparams);
            }
        }
    }
    _w.closeTag();
}

// _____________________________________________________________________________
void SvgRenderer::renderNodeFronts(const RenderGraph& outG,
                                   const RenderParams& rparams) {
  _w.openTag("g");
  for (auto n : outG.getNds()) {
    std::string color = n->pl().stops().size() > 0 ? "red" : "black";
    for (auto& f : n->pl().fronts()) {
      const PolyLine<double> p = f.geom;
      std::stringstream style;
      style << "fill:none;stroke:" << color
            << ";stroke-linejoin: "
               "miter;stroke-linecap:round;stroke-opacity:0.9;stroke-width:1";
      std::map<std::string, std::string> params;
      params["style"] = style.str();
      printLine(p, params, rparams);

      DPoint a = p.getPointAt(.5).p;

      std::stringstream styleA;
      styleA << "fill:none;stroke:" << color
             << ";stroke-linejoin: "
                "miter;stroke-linecap:round;stroke-opacity:1;stroke-width:.5";
      params["style"] = styleA.str();

      printLine(PolyLine<double>(*n->pl().getGeom(), a), params, rparams);
    }
  }
  _w.closeTag();
}

// _____________________________________________________________________________
void SvgRenderer::outputEdges(const RenderGraph& outG,
                              const RenderParams& rparams) {
  struct cmp {
    bool operator()(const LineNode* lhs, const LineNode* rhs) const {
      return lhs->getAdjList().size() > rhs->getAdjList().size() ||
             (lhs->getAdjList().size() == rhs->getAdjList().size() &&
              RenderGraph::getConnCardinality(lhs) >
                  RenderGraph::getConnCardinality(rhs)) ||
             (lhs->getAdjList().size() == rhs->getAdjList().size() &&
              lhs > rhs);
    }
  };

  struct cmpEdge {
    bool operator()(const shared::linegraph::LineEdge* lhs,
                    const shared::linegraph::LineEdge* rhs) const {
      return lhs->pl().getLines().size() < rhs->pl().getLines().size() ||
             (lhs->pl().getLines().size() == rhs->pl().getLines().size() &&
              lhs < rhs);
    }
  };

  std::set<const LineNode*, cmp> nodesOrdered;
  std::set<const shared::linegraph::LineEdge*, cmpEdge> edgesOrdered;
  for (auto nd : outG.getNds()) nodesOrdered.insert(nd);

  std::set<const shared::linegraph::LineEdge*> rendered;

  for (const auto n : nodesOrdered) {
    edgesOrdered.insert(n->getAdjList().begin(), n->getAdjList().end());

    for (const auto* e : edgesOrdered) {
      if (rendered.insert(e).second) renderEdgeTripGeom(outG, e, rparams);
    }
  }
}

// _____________________________________________________________________________
void SvgRenderer::renderNodeConnections(const RenderGraph& outG,
                                        const LineNode* n,
                                        const RenderParams& rparams) {
  UNUSED(rparams);
  auto geoms = outG.innerGeoms(n, _cfg->innerGeometryPrecision);

  for (auto& clique : getInnerCliques(n, geoms, 9999)) renderClique(clique, n);
}

// _____________________________________________________________________________
std::multiset<InnerClique> SvgRenderer::getInnerCliques(
    const shared::linegraph::LineNode* n, std::vector<InnerGeom> pool,
    size_t level) const {
  std::multiset<InnerClique> ret;

  // start with the first geom in pool
  while (!pool.empty()) {
    InnerClique cur(n, pool.front());
    pool.erase(pool.begin());

    size_t p;
    while ((p = getNextPartner(cur, pool, level)) < pool.size()) {
      cur.geoms.push_back(pool[p]);
      pool.erase(pool.begin() + p);
    }

    ret.insert(cur);
  }

  return ret;
}

// _____________________________________________________________________________
size_t SvgRenderer::getNextPartner(const InnerClique& forClique,
                                   const std::vector<InnerGeom>& pool,
                                   size_t level) const {
  for (size_t i = 0; i < pool.size(); i++) {
    const auto& ic = pool[i];
    for (auto& ciq : forClique.geoms) {
      if (isNextTo(ic, ciq) || (level > 1 && hasSameOrigin(ic, ciq))) {
        return i;
      }
    }
  }

  return pool.size();
}

// _____________________________________________________________________________
bool SvgRenderer::isNextTo(const InnerGeom& a, const InnerGeom& b) const {
  double THRESHOLD = 0.5 * M_PI + 0.1;

  if (!a.from.edge) return false;
  if (!b.from.edge) return false;
  if (!a.to.edge) return false;
  if (!b.to.edge) return false;

  auto nd = RenderGraph::sharedNode(a.from.edge, a.to.edge);

  assert(a.from.edge);
  assert(b.from.edge);
  assert(a.to.edge);
  assert(b.to.edge);

  bool aFromInv = a.from.edge->getTo() == nd;
  bool bFromInv = b.from.edge->getTo() == nd;
  bool aToInv = a.to.edge->getTo() == nd;
  bool bToInv = b.to.edge->getTo() == nd;

  int aSlotFrom = !aFromInv
                      ? a.slotFrom
                      : (a.from.edge->pl().getLines().size() - 1 - a.slotFrom);
  int aSlotTo =
      !aToInv ? a.slotTo : (a.to.edge->pl().getLines().size() - 1 - a.slotTo);
  int bSlotFrom = !bFromInv
                      ? b.slotFrom
                      : (b.from.edge->pl().getLines().size() - 1 - b.slotFrom);
  int bSlotTo =
      !bToInv ? b.slotTo : (b.to.edge->pl().getLines().size() - 1 - b.slotTo);

  if (a.from.edge == b.from.edge && a.to.edge == b.to.edge) {
    if ((aSlotFrom - bSlotFrom == 1 && bSlotTo - aSlotTo == 1) ||
        (bSlotFrom - aSlotFrom == 1 && aSlotTo - bSlotTo == 1)) {
      return true;
      double ang1 = fabs(util::geo::angBetween(a.geom.front(), a.geom.back()));
      double ang2 = fabs(util::geo::angBetween(b.geom.front(), b.geom.back()));

      return ang1 > THRESHOLD && ang2 > THRESHOLD;
    }
  }

  if (a.to.edge == b.from.edge && a.from.edge == b.to.edge) {
    if ((aSlotFrom - bSlotTo == 1 && bSlotFrom - aSlotTo == 1) ||
        (bSlotTo - aSlotFrom == 1 && aSlotTo - bSlotFrom == 1)) {
      return true;
      double ang1 = fabs(util::geo::angBetween(a.geom.front(), a.geom.back()));
      double ang2 = fabs(util::geo::angBetween(b.geom.front(), b.geom.back()));

      return ang1 > THRESHOLD && ang2 > THRESHOLD;
    }
  }

  return false;
}

// _____________________________________________________________________________
bool SvgRenderer::hasSameOrigin(const InnerGeom& a, const InnerGeom& b) const {
  if (a.from.edge == b.from.edge) {
    return a.slotFrom == b.slotFrom;
  }
  if (a.to.edge == b.from.edge) {
    return a.slotTo == b.slotFrom;
  }
  if (a.to.edge == b.to.edge) {
    return a.slotTo == b.slotTo;
  }
  if (a.from.edge == b.to.edge) {
    return a.slotFrom == b.slotTo;
  }

  return false;
}

// _____________________________________________________________________________
void SvgRenderer::renderClique(const InnerClique& cc, const LineNode* n) {
  _innerDelegates.push_back(
      std::map<uintptr_t, std::vector<OutlinePrintPair>>());
  std::multiset<InnerClique> renderCliques = getInnerCliques(n, cc.geoms, 0);
  for (const auto& c : renderCliques) {
    // the longest geom will be the ref geom
    InnerGeom ref = c.geoms[0];
    for (size_t i = 1; i < c.geoms.size(); i++) {
      if (c.geoms[i].geom.getLength() > ref.geom.getLength()) ref = c.geoms[i];
    }

    for (size_t i = 0; i < c.geoms.size(); i++) {
      PolyLine<double> pl = c.geoms[i].geom;

      if (ref.geom.getLength() >
          (_cfg->lineWidth + 2 * 0.9* _cfg->outlineWidth + _cfg->lineSpacing)) { // did times 0.5
        double off =
            -(_cfg->lineWidth + _cfg->lineSpacing + 2 * _cfg->outlineWidth) *
            (static_cast<int>(c.geoms[i].slotFrom) -
             static_cast<int>(ref.slotFrom));

        if (ref.from.edge->getTo() == n) off = -off;

        pl = ref.geom.offsetted(off);

        if (pl.getLength() / c.geoms[i].geom.getLength() > 1.5)
          pl = c.geoms[i].geom;

        std::set<LinePoint<double>, LinePointCmp<double>> a;
        std::set<LinePoint<double>, LinePointCmp<double>> b;

        if (ref.from.edge)
          a = n->pl().frontFor(ref.from.edge)->geom.getIntersections(pl);
        if (ref.to.edge)
          b = n->pl().frontFor(ref.to.edge)->geom.getIntersections(pl);

        if (a.size() > 0 && b.size() > 0) {
          pl = pl.getSegment(a.begin()->totalPos, b.begin()->totalPos);
        } else if (a.size() > 0) {
          pl = pl.getSegment(a.begin()->totalPos, 1);
        } else if (b.size() > 0) {
          pl = pl.getSegment(0, b.begin()->totalPos);
        }
      }

      double strokeWidth = _cfg->lineWidth * _cfg->outputResolution;

std::string color = c.geoms[i].from.line->color();
// if (color == "FF0000") {       // red
//     strokeWidth *= 3;
// } else if (color == "228B22") { // green
//     strokeWidth *= 2.25;
// } else if (color == "0000FF") { // blue
//     strokeWidth *= 1.5;
// } else if (color == "FF00FF") { // fuchsia
//     strokeWidth *= 3;
// }
if (color == "FF0000") {       // red
    strokeWidth *= 1;
} else if (color == "228B22") { // green
    strokeWidth *= 0.9;
} else if (color == "0000FF") { // blue
    strokeWidth *= 0.8;
} else if (color == "FF00FF") { // fuchsia
    strokeWidth *= 1;
} else if (color == "000000") { // black
    strokeWidth *= 0.7;
}





      std::stringstream styleOutlineCropped;
      styleOutlineCropped << "fill:none;stroke:#000000";

      styleOutlineCropped << ";stroke-linecap:butt;stroke-width:"
                          << (_cfg->lineWidth + _cfg->outlineWidth) * 0.9 *
                                 _cfg->outputResolution; //added 0.5
      Params paramsOutlineCropped;
      paramsOutlineCropped["style"] = styleOutlineCropped.str();
      paramsOutlineCropped["class"] += " inner-geom-outline";
      paramsOutlineCropped["class"] +=
          " " + getLineClass(c.geoms[i].from.line->id());

std::stringstream styleStr;
styleStr << "fill:none;stroke:#" << color;
styleStr << ";stroke-linecap:round;stroke-opacity:1;stroke-width:" 
         << strokeWidth;
         
      Params params;
      params["style"] = styleStr.str();
      params["class"] += " inner-geom ";
      params["class"] += " " + getLineClass(c.geoms[i].from.line->id());

      _innerDelegates.back()[(uintptr_t)c.geoms[i].from.line].push_back(
          OutlinePrintPair(PrintDelegate(params, pl),
                           PrintDelegate(paramsOutlineCropped, pl)));
    }
  }
}

// _____________________________________________________________________________
void SvgRenderer::renderLinePart(const PolyLine<double> p, double width,
                                 const Line& line, const std::string& css,
                                 const std::string& oCss) {
  renderLinePart(p, width, line, css, oCss, "");
}

// _____________________________________________________________________________
void SvgRenderer::renderLinePart(const PolyLine<double> p, double width,
                                 const Line& line, const std::string& css,
                                 const std::string& oCss,
                                 const std::string& endMarker) {
    // ---- Outline style ----
    std::stringstream styleOutline;
    styleOutline << "fill:none;stroke:#000000;stroke-linecap:round;stroke-width:"
                 << (width + _cfg->outlineWidth) * 0.9 * _cfg->outputResolution << ";" //made outline * 0.5 to make smaller
                 << oCss;

    Params paramsOutline;
    paramsOutline["style"] = styleOutline.str();
    paramsOutline["class"] = "transit-edge-outline " + getLineClass(line.id());

    // ---- Compute stroke width ----
    double strokeWidth = width * _cfg->outputResolution;
   
    if (line.color() == "FF0000") {       // red
        strokeWidth *= 1;
    } else if (line.color() == "228B22") { // green
        strokeWidth *= 0.9;
    } else if (line.color() == "0000FF") { // blue
        strokeWidth *= 0.8;
    } else if (line.color() == "FF00FF") { // fuchsia
        strokeWidth *= 1;
    } else if (line.color() == "000000") { // black
        strokeWidth *= 0.7;
    }

    // ---- Main line style ----
    std::stringstream styleStr;
    styleStr << "fill:none;stroke:#" << line.color() << ";" << css;

    if (!endMarker.empty()) {
        styleStr << ";marker-end:url(#" << endMarker << ")";
    }

    styleStr << ";stroke-linecap:round;stroke-opacity:1;stroke-width:" << strokeWidth;

    Params params;
    params["style"] = styleStr.str();
    params["class"] = "transit-edge " + getLineClass(line.id());

    // ---- Insert into delegates ----
    _delegates[0].insert(_delegates[0].begin(),
                         OutlinePrintPair(PrintDelegate(params, p),
                                          PrintDelegate(paramsOutline, p)));
}


// _____________________________________________________________________________
void SvgRenderer::renderEdgeTripGeom(const RenderGraph& outG,
                                     const shared::linegraph::LineEdge* e,
                                     const RenderParams& rparams) {
  UNUSED(rparams);
  const shared::linegraph::NodeFront* nfTo = e->getTo()->pl().frontFor(e);
  const shared::linegraph::NodeFront* nfFrom = e->getFrom()->pl().frontFor(e);

  assert(nfTo);
  assert(nfFrom);

  PolyLine<double> center(*e->pl().getGeom());

  double lineW = _cfg->lineWidth;
  double outlineW = _cfg->outlineWidth;
  double lineSpc = _cfg->lineSpacing;
  double offsetStep = lineW + 2.0 * outlineW + lineSpc;
  double oo = outG.getTotalWidth(e);

  double o = oo;

  for (size_t i = 0; i < e->pl().getLines().size(); i++) {
    const auto& lo = e->pl().lineOccAtPos(i);

    const Line* line = lo.line;
    PolyLine<double> p = center;

    if (p.getLength() < 0.01) continue;

    double offset = -(o - oo / 2.0 - (2.0 * outlineW + _cfg->lineWidth) / 2.0);

    p.offsetPerp(offset);

    auto iSects = nfTo->geom.getIntersections(p);
    if (iSects.size() > 0) {
      p = p.getSegment(0, iSects.begin()->totalPos);
    } else {
      p << nfTo->geom.projectOn(p.back()).p;
    }

    auto iSects2 = nfFrom->geom.getIntersections(p);
    if (iSects2.size() > 0) {
      p = p.getSegment(iSects2.begin()->totalPos, 1);
    } else {
      p >> nfFrom->geom.projectOn(p.front()).p;
    }

    double arrowLength = (_cfg->lineWidth * 2.5);

    std::string css, oCss;

    if (!lo.style.isNull()) {
      css = lo.style.get().getCss();
      oCss = lo.style.get().getOutlineCss();
    }

    if (_cfg->renderDirMarkers && lo.direction != 0 &&
        center.getLength() > arrowLength * 3) {
      std::stringstream markerName;
      markerName << e << ":" << line << ":" << i;

      std::string markerPathMale = getMarkerPathMale(lineW);
      EndMarker emm(markerName.str() + "_m", "white", markerPathMale, lineW,
                    lineW);

      _markers.push_back(emm);

      PolyLine<double> firstPart = p.getSegmentAtDist(0, p.getLength() / 2);
      PolyLine<double> secondPart =
          p.getSegmentAtDist(p.getLength() / 2, p.getLength());

      if (lo.direction == e->getTo()) {
        renderLinePart(firstPart, lineW, *line, css, oCss,
                       markerName.str() + "_m");
        renderLinePart(secondPart.reversed(), lineW, *line, css, oCss);
      } else {
        renderLinePart(secondPart.reversed(), lineW, *line, css, oCss,
                       markerName.str() + "_m");
        renderLinePart(firstPart, lineW, *line, css, oCss);
      }
    } else {
      renderLinePart(p, lineW, *line, css, oCss);
    }

    o -= offsetStep;
  }
}

// _____________________________________________________________________________
std::string SvgRenderer::getMarkerPathMale(double w) const {
  UNUSED(w);
  return "M0,0 V1 H.5 L1.3,.5 L.5,0 Z";
}

// _____________________________________________________________________________
void SvgRenderer::renderDelegates(const RenderGraph& outG,
                                  const RenderParams& rparams) {
  UNUSED(outG);
  for (auto& a : _delegates) {
    _w.openTag("g");
    for (auto& pd : a.second) {
      if (_cfg->outlineWidth > 0) {
        printLine(pd.back.second, pd.back.first, rparams);
      }
      printLine(pd.front.second, pd.front.first, rparams);
    }
    _w.closeTag();
  }

  for (auto& a : _innerDelegates) {
    _w.openTag("g");
    for (auto& b : a) {
      for (auto& pd : b.second) {
        if (_cfg->outlineWidth > 0) {
          printLine(pd.back.second, pd.back.first, rparams);
        }
      }
      for (auto& pd : b.second) {
        printLine(pd.front.second, pd.front.first, rparams);
      }
    }
    _w.closeTag();
  }
}

// _____________________________________________________________________________
void SvgRenderer::printPoint(const DPoint& p, const std::string& style,
                             const RenderParams& rparams) {
  std::map<std::string, std::string> params;
  params["cx"] =
      std::to_string((p.getX() - rparams.xOff) * _cfg->outputResolution);
  params["cy"] = std::to_string(rparams.height - (p.getY() - rparams.yOff) *
                                                     _cfg->outputResolution);
  params["r"] = "2";
  params["style"] = style;
  _w.openTag("circle", params);
  _w.closeTag();
}

// _____________________________________________________________________________
void SvgRenderer::printLine(const PolyLine<double>& l, const std::string& style,
                            const RenderParams& rparams) {
  std::map<std::string, std::string> params;
  params["style"] = style;
  printLine(l, params, rparams);
}

// _____________________________________________________________________________
void SvgRenderer::printLine(const PolyLine<double>& l,
                            const std::map<std::string, std::string>& ps,
                            const RenderParams& rparams) {
  std::map<std::string, std::string> params = ps;
  std::stringstream points;

  for (auto& p : l.getLine()) {
    points << " " << (p.getX() - rparams.xOff) * _cfg->outputResolution << ","
           << rparams.height -
                  (p.getY() - rparams.yOff) * _cfg->outputResolution;
  }

  params["points"] = points.str();

  _w.openTag("polyline", params);
  _w.closeTag();
}

// _____________________________________________________________________________
void SvgRenderer::printPolygon(const Polygon<double>& g,
                               const std::map<std::string, std::string>& ps,
                               const RenderParams& rparams) {
  std::map<std::string, std::string> params = ps;
  std::stringstream points;

  for (auto& p : g.getOuter()) {
    points << " " << (p.getX() - rparams.xOff) * _cfg->outputResolution << ","
           << rparams.height -
                  (p.getY() - rparams.yOff) * _cfg->outputResolution;
  }

  params["points"] = points.str();
  params["class"] = "station-poly";

  _w.openTag("polygon", params);
  _w.closeTag();
}

// _____________________________________________________________________________
void SvgRenderer::printCircle(const DPoint& center, double rad,
                              const std::string& style,
                              const RenderParams& rparams) {
  std::map<std::string, std::string> params;
  params["style"] = style;
  printCircle(center, rad, params, rparams);
}

// _____________________________________________________________________________
void SvgRenderer::printCircle(const DPoint& center, double rad,
                              const std::map<std::string, std::string>& ps,
                              const RenderParams& rparams) {
  std::map<std::string, std::string> params = ps;
  std::stringstream points;

  params["cx"] =
      std::to_string((center.getX() - rparams.xOff) * _cfg->outputResolution);
  params["cy"] = std::to_string(
      rparams.height - (center.getY() - rparams.yOff) * _cfg->outputResolution);
  params["r"] = std::to_string(rad * _cfg->outputResolution);

  _w.openTag("circle", params);
  _w.closeTag();
}

// _____________________________________________________________________________
size_t InnerClique::getNumBranchesIn(
    const shared::linegraph::LineEdge* edg) const {
  std::set<size_t> slots;
  size_t ret = 0;
  for (const auto& ig : geoms) {
    if (ig.from.edge == edg && !slots.insert(ig.slotFrom).second) ret++;
    if (ig.to.edge == edg && !slots.insert(ig.slotTo).second) ret++;
  }

  return ret;
}

// _____________________________________________________________________________
void SvgRenderer::renderStationLabels(const Labeller& labeller,
                                      const RenderParams& rparams) {
  _w.openTag("g");
  size_t id = 0;


  for (auto label : labeller.getStationLabels()) {
    // added
    const std::string& stationName = label.s.name;
    // skip labels where third character is a digit
    if (stationName.size() >= 3 && std::isdigit(stationName[2])) {
        std::cerr << "Skipping label for station: " << stationName << std::endl;
        continue;
    }
    // end added

    std::string shift = "0em";
    std::string textAnchor = "start";
    std::string startOffset = "0";
    auto textPath = label.geom;
    double ang = util::geo::angBetween(textPath.front(), textPath.back());

    if ((fabs(ang) < (3 * M_PI / 2)) && (fabs(ang) > (M_PI / 2))) {
      shift = ".75em";
      textAnchor = "end";
      startOffset = "100%";
      textPath.reverse();
    }

    std::stringstream points;
    std::map<std::string, std::string> pathPars;

    points << "M"
           << (textPath.front().getX() - rparams.xOff) * _cfg->outputResolution
           << " "
           << rparams.height - (textPath.front().getY() - rparams.yOff) *
                                   _cfg->outputResolution;

    for (auto& p : textPath.getLine()) {
      points << " L" << (p.getX() - rparams.xOff) * _cfg->outputResolution
             << " "
             << rparams.height -
                    (p.getY() - rparams.yOff) * _cfg->outputResolution;
    }

    std::string idStr = "stlblp" + util::toString(id);

    pathPars["d"] = points.str();
    pathPars["id"] = idStr;
    id++;

    _w.openTag("defs");
    _w.openTag("path", pathPars);
    _w.closeTag();
    _w.closeTag();

    std::map<std::string, std::string> params;
    params["class"] = "station-label";
    params["font-weight"] = label.bold ? "normal" : "normal";
    params["font-family"] = "London Tube";
    params["dy"] = shift;
    params["font-size"] =
        util::toString(label.fontSize * _cfg->outputResolution);

    _w.openTag("text", params);
    _w.openTag("textPath", {{"dy", shift},
                            {"xlink:href", "#" + idStr},
                            {"startOffset", startOffset},
                            {"text-anchor", textAnchor}});

    _w.writeText(label.s.name);
    _w.closeTag();
    _w.closeTag();
  }
  _w.closeTag();
}




// _____________________________________________________________________________
void SvgRenderer::renderLineLabels(const Labeller& labeller,
                                   const RenderParams& rparams) {
  _w.openTag("g");
  size_t id = 0;
  for (auto label : labeller.getLineLabels()) {
    std::string shift = "0em";
    auto textPath = label.geom;
    double ang = util::geo::angBetween(textPath.front(), textPath.back());

    if ((fabs(ang) < (3 * M_PI / 2)) && (fabs(ang) > (M_PI / 2))) {
      shift = ".75em";
      textPath.reverse();
    }

    std::stringstream points;
    std::map<std::string, std::string> pathPars;

    points << "M"
           << (textPath.front().getX() - rparams.xOff) * _cfg->outputResolution
           << " "
           << rparams.height - (textPath.front().getY() - rparams.yOff) *
                                   _cfg->outputResolution;

    for (auto& p : textPath.getLine()) {
      points << " L" << (p.getX() - rparams.xOff) * _cfg->outputResolution
             << " "
             << rparams.height -
                    (p.getY() - rparams.yOff) * _cfg->outputResolution;
    }

    std::string idStr = "textp" + util::toString(id);

    pathPars["d"] = points.str();
    pathPars["id"] = idStr;
    id++;

    _w.openTag("defs");
    _w.openTag("path", pathPars);
    _w.closeTag();
    _w.closeTag();

    std::map<std::string, std::string> params;
    params["class"] = "line-label";
    params["font-weight"] = "bold";
    params["font-family"] = "Ubuntu";
    params["dy"] = shift;
    params["font-size"] =
        util::toString(label.fontSize * _cfg->outputResolution);

    _w.openTag("text", params);
    _w.openTag("textPath", {{"dy", shift},
                            {"xlink:href", "#" + idStr},
                            {"text-anchor", "middle"},
                            {"startOffset", "50%"}});

    double dy = 0;
    for (auto line : label.lines) {
      _w.openTag("tspan",
                 {{"fill", "#" + line->color()}, {"dx", util::toString(dy)}});
      dy = (label.fontSize * _cfg->outputResolution) / 3;
      _w.writeText(line->label());
      _w.closeTag();
    }
    _w.closeTag();
    _w.closeTag();
  }
  _w.closeTag();
}

// _____________________________________________________________________________
double InnerClique::getZWeight() const {
  // more weight = more to the bottom

  double BRANCH_WEIGHT = 4;

  double ret = 0;

  ret = geoms.size();  // baseline: threads with more lines to the bottom,
                       // because they are easier to follow

  for (const auto& nf : n->pl().fronts()) {
    ret -= getNumBranchesIn(nf.edge) * BRANCH_WEIGHT;
  }

  return ret;
}

// _____________________________________________________________________________
std::string SvgRenderer::getLineClass(const std::string& id) const {
  auto i = lineClassIds.find(id);
  if (i != lineClassIds.end()) return "line-" + std::to_string(i->second);

  lineClassIds[id] = ++lineClassId;
  return "line-" + std::to_string(lineClassId);
}

// _____________________________________________________________________________
bool InnerClique::operator<(const InnerClique& rhs) const {
  // more weight = more to the bottom
  return getZWeight() > rhs.getZWeight();
}
